In [ ]:
import os
import re
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.impute import SimpleImputer
from sklearn.ensemble import ExtraTreesClassifier
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    precision_recall_fscore_support
)

warnings.filterwarnings("ignore")

try:
    import shap
except ModuleNotFoundError:
    raise ModuleNotFoundError(
        "The shap package is not installed in the current environment. "
        "Please run: pip install shap"
    )


# ============================================================
# 0. Path and parameter configuration
# ============================================================

def find_project_root():
    """
    Locate the project root from either a Python script or a notebook.

    Expected project structure:
    Project_root/
    ├─ Code/
    ├─ Data/
    └─ Result/
    """
    if "__file__" in globals():
        start_dir = os.path.abspath(os.path.dirname(__file__))
    else:
        start_dir = os.path.abspath(os.getcwd())

    current = start_dir

    while True:
        if (
            os.path.exists(os.path.join(current, "Code"))
            and os.path.exists(os.path.join(current, "Data"))
        ):
            return current

        parent = os.path.dirname(current)

        if parent == current:
            break

        current = parent

    if os.path.basename(start_dir).lower() == "classifier":
        return os.path.dirname(os.path.dirname(start_dir))

    if os.path.basename(start_dir).lower() == "code":
        return os.path.dirname(start_dir)

    return start_dir


PROJECT_ROOT = find_project_root()

RESULT_BASE_DIR = os.path.join(PROJECT_ROOT, "Result")

DATA_ROOT = os.path.join(
    RESULT_BASE_DIR,
    "01_Foldwise_Preprocessing_and_Ratio_Features"
)

STABILITY_SUMMARY_FILE = os.path.join(
    RESULT_BASE_DIR,
    "05_Stable_Cluster_Champions_and_Novel_Ratio_Candidates",
    "rho090_stable_champions_and_ratio_candidates_summary.xlsx"
)

OUT_DIR = os.path.join(
    RESULT_BASE_DIR,
    "11_Final_Model_SHAP_Interpretation"
)
os.makedirs(OUT_DIR, exist_ok=True)

TYPE_COL = "Type"
N_OUTER_FOLDS = 5

# The main-text SHAP interpretation uses the KNN-imputation workflow.
# If global_mean is needed for supplementary analysis, change this to:
# SHAP_METHODS = ["global_mean", "knn"]
SHAP_METHODS = ["knn"]

CLASS_ORDER = ["A", "S", "I"]

MODEL_NAME = "ExtraTrees"
FEATURE_SET_NAME = "Stable_features"

RANDOM_STATE = 42
N_ESTIMATORS = 600

TOP_N_GLOBAL = 20
TOP_N_CLASSWISE = 15

# Maximum number of samples for SHAP calculation.
# Set this to None to calculate SHAP values for all samples.
SHAP_MAX_SAMPLES = None

# Six representative dependence plots for the main text.
REPRESENTATIVE_FEATURES = [
    "R_Trace_Yb/Lu",
    "R_Trace_Dy/Ho",
    "R_Trace_Nb/Pb",
    "R_Major_TiO2/P2O5",
    "R_Major_Fe2O3t/P2O5",
    "R_Major_Al2O3/K2O",
]


# ============================================================
# 1. Utility functions
# ============================================================

def normalize_type_value(v):
    s = str(v).strip()

    if s in ["A", "A-type", "A-Type", "A_TYPE", "A type"]:
        return "A"
    if s in ["S", "S-type", "S-Type", "S_TYPE", "S type"]:
        return "S"
    if s in ["I", "I-type", "I-Type", "I_TYPE", "I type"]:
        return "I"

    return s


def display_feature_name(name):
    s = str(name)

    s = s.replace("R_Major_", "")
    s = s.replace("R_Trace_", "")

    s = s.replace("A12O3", "Al2O3")
    s = s.replace("10000*Ga/A1", "10000×Ga/Al")
    s = s.replace("10000*Ga/Al", "10000×Ga/Al")

    return s


def safe_filename(name):
    s = display_feature_name(name)
    s = re.sub(r"[\\/:*?\"<>|]", "_", s)
    s = s.replace("×", "x")
    s = s.replace("+", "plus")
    s = s.replace(" ", "_")
    return s


def feature_key(name):
    s = str(name).strip()
    s = display_feature_name(s)
    s = s.replace("×", "*")
    s = s.replace("：", ":")
    s = re.sub(r"\s+", "", s)
    return s.lower()


def resolve_one_feature(feature, columns):
    columns = [str(c).strip() for c in columns]

    if feature in columns:
        return feature

    target_key = feature_key(feature)

    for c in columns:
        if feature_key(c) == target_key:
            return c

    # Fix possible Al2O3 / A12O3 variants.
    fixed_candidates = [
        str(feature).replace("A12O3", "Al2O3"),
        str(feature).replace("Al2O3", "A12O3"),
    ]

    for fixed in fixed_candidates:
        if fixed in columns:
            return fixed

        fixed_key = feature_key(fixed)

        for c in columns:
            if feature_key(c) == fixed_key:
                return c

    return None


def resolve_feature_list(features, columns):
    resolved = []
    missing = []

    for f in features:
        c = resolve_one_feature(f, columns)

        if c is None:
            missing.append(f)
        else:
            if c not in resolved:
                resolved.append(c)

    return resolved, missing


def clean_X(df, features):
    X = df[features].copy()

    for c in X.columns:
        X[c] = pd.to_numeric(X[c], errors="coerce")

    X = X.replace([np.inf, -np.inf], np.nan)

    return X


def remove_bad_features(X):
    """
    Remove all-NaN or constant columns.

    This is used only for the final interpretation model and does not
    affect performance evaluation.
    """
    keep_cols = []

    for c in X.columns:
        x = pd.to_numeric(X[c], errors="coerce")
        x = x.replace([np.inf, -np.inf], np.nan)
        non_na = x.dropna()

        if len(non_na) == 0:
            continue

        if non_na.nunique() <= 1:
            continue

        keep_cols.append(c)

    return X[keep_cols].copy(), keep_cols


def load_stable_features_for_paper():
    if not os.path.exists(STABILITY_SUMMARY_FILE):
        raise FileNotFoundError(
            f"Script 05 summary file was not found: {STABILITY_SUMMARY_FILE}"
        )

    xls = pd.ExcelFile(STABILITY_SUMMARY_FILE)

    preferred_sheets = [
        "stable_features_for_paper",
        "core_stable_features",
        "champion_stability_summary",
        "stable_champion_features",
    ]

    for sheet in preferred_sheets:
        if sheet in xls.sheet_names:
            df = pd.read_excel(STABILITY_SUMMARY_FILE, sheet_name=sheet)

            if "Feature" in df.columns:
                features = (
                    df["Feature"]
                    .dropna()
                    .astype(str)
                    .tolist()
                )

                if len(features) > 0:
                    print(f"Stable features were read from sheet: {sheet}")
                    print(f"Number of stable features: {len(features)}")
                    return features

    raise ValueError(
        "No usable Feature table was found in the Script 05 summary file. "
        "Expected sheets include stable_features_for_paper, "
        "champion_stability_summary, or related stable-feature sheets."
    )


def read_oof_full_dataset(method):
    """
    Concatenate the five outer-fold test files.

    Each sample appears only once in the out-of-fold dataset.
    This dataset is used for final interpretation figures, not for
    reporting generalization performance.
    """
    dfs = []

    for fold in range(1, N_OUTER_FOLDS + 1):
        test_path = os.path.join(
            DATA_ROOT,
            method,
            f"fold_{fold:02d}_test_with_ratios.xlsx"
        )

        if not os.path.exists(test_path):
            raise FileNotFoundError(test_path)

        df = pd.read_excel(test_path)
        df.columns = [str(c).strip() for c in df.columns]

        if TYPE_COL not in df.columns:
            raise ValueError(f"The label column is missing: {TYPE_COL}")

        df[TYPE_COL] = df[TYPE_COL].apply(normalize_type_value)
        df["Outer_fold"] = fold
        df["Imputation_method"] = method

        dfs.append(df)

    full_df = pd.concat(dfs, axis=0, ignore_index=True)

    print(f"\n{method} OOF concatenated dataset:")
    print("Sample count:", len(full_df))
    print("Class distribution:")
    print(full_df[TYPE_COL].value_counts())

    return full_df


def make_final_model():
    model = ExtraTreesClassifier(
        n_estimators=N_ESTIMATORS,
        max_features="sqrt",
        random_state=RANDOM_STATE,
        n_jobs=-1,
        class_weight=None
    )

    return model


def normalize_shap_values(raw_shap_values, n_samples, n_features, n_classes):
    """
    Normalize SHAP output formats from different shap versions.

    Final shape:
    shap_arr.shape = (n_samples, n_features, n_classes)
    """
    if isinstance(raw_shap_values, list):
        # List length = n_classes, each array = (n_samples, n_features).
        shap_arr = np.stack(raw_shap_values, axis=2)
        return shap_arr

    arr = np.asarray(raw_shap_values)

    if arr.ndim == 2:
        # Binary-class style output may be (n_samples, n_features).
        arr = arr[:, :, np.newaxis]
        return arr

    if arr.ndim == 3:
        # Common new format: (n_samples, n_features, n_classes).
        if arr.shape[0] == n_samples and arr.shape[1] == n_features:
            return arr

        # Possible old format: (n_classes, n_samples, n_features).
        if arr.shape[0] == n_classes and arr.shape[1] == n_samples:
            arr = np.transpose(arr, (1, 2, 0))
            return arr

    raise ValueError(
        f"Unrecognized SHAP output shape: {arr.shape}; "
        f"n_samples={n_samples}, n_features={n_features}, n_classes={n_classes}"
    )


def classify_feature_group(feature):
    s = str(feature)
    body = display_feature_name(s)

    classical_names = {
        "10000×Ga/Al",
        "A/CNK",
        "A/NK",
        "Zr+Nb+Ce+Y",
        "Sr/Y",
        "Rb/Sr",
        "K2O/Na2O",
        "Fe2O3t/MgO",
    }

    if body in classical_names:
        return "Classical geochemical index"

    if s.startswith("R_Major_"):
        return "Major-element ratio"

    if not s.startswith("R_Trace_"):
        return "Original element or classical feature"

    hfse = {"Nb", "Ta", "Zr", "Hf", "Ti", "Y"}
    ree = {
        "La", "Ce", "Pr", "Nd", "Sm", "Eu", "Gd",
        "Tb", "Dy", "Ho", "Er", "Tm", "Yb", "Lu"
    }
    lile = {"Rb", "Sr", "Ba", "Cs", "Pb"}
    thu = {"Th", "U"}

    parts = re.split(r"[/_+\-*()]+", body)
    parts = [p for p in parts if p]

    has_hfse = any(p in hfse for p in parts)
    has_ree = any(p in ree for p in parts)
    has_lile = any(p in lile for p in parts)
    has_thu = any(p in thu for p in parts)

    if has_hfse and has_ree:
        return "HFSE-REE ratio"
    if has_hfse and has_thu:
        return "HFSE-Th-U ratio"
    if has_ree:
        return "REE fractionation ratio"
    if has_hfse:
        return "HFSE-related ratio"
    if has_lile:
        return "LILE-related ratio"

    return "Trace-element ratio"


def interpretation_note(feature):
    name = display_feature_name(feature)

    notes = {
        "10000×Ga/Al": "Classical A-type granite discriminator related to Ga enrichment and Al depletion.",
        "A/CNK": "Aluminosity index commonly used to distinguish peraluminous S-type affinity.",
        "A/NK": "Alkali-normalized aluminosity index.",
        "Zr+Nb+Ce+Y": "Classical HFSE-REE enrichment index for A-type granite discrimination.",
        "Sr/Y": "Indicator related to plagioclase/amphibole fractionation and source/depth effects.",
        "Rb/Sr": "Index related to differentiation, feldspar fractionation, and crustal evolution.",
        "Yb/Lu": "Heavy REE fractionation signal.",
        "Ho/Er": "Middle-to-heavy REE fractionation signal.",
        "Dy/Ho": "HREE fractionation and REE pattern curvature signal.",
        "Y/Dy": "Y-HREE behavior and fractionation signal.",
        "Y/Er": "Y-HREE behavior and fractionation signal.",
        "Sm/Gd": "Middle REE fractionation signal.",
        "Nb/Pb": "HFSE vs crustal/LILE-related enrichment contrast.",
        "Ta/Th": "HFSE-Th contrast related to source and differentiation effects.",
        "Nb/U": "HFSE-U contrast, potentially reflecting crustal affinity and source differences.",
        "TiO2/MgO": "Major-element ratio related to mafic mineral and Fe-Ti oxide control.",
        "TiO2/P2O5": "Fe-Ti-P system, possibly linked to oxide and apatite fractionation.",
        "Fe2O3t/P2O5": "Fe-P differentiation and accessory mineral control.",
        "Al2O3/K2O": "Aluminous and feldspar/mica-related signal.",
    }

    return notes.get(name, "")


# ============================================================
# 2. Plotting functions
# ============================================================

def bold_tick_labels(ax):
    """
    Bold x/y tick labels only, without changing font size, color, layout,
    or other plotting styles.
    """
    for label in ax.get_xticklabels():
        label.set_fontweight("bold")

    for label in ax.get_yticklabels():
        label.set_fontweight("bold")


def plot_global_shap_bar(global_imp_df, out_png, top_n=20):
    sub = global_imp_df.head(top_n).copy()
    sub = sub.sort_values("Mean_abs_SHAP_overall", ascending=True)

    plt.figure(figsize=(8.5, max(5.5, 0.35 * len(sub))), dpi=300)

    plt.barh(
        sub["Display_feature"],
        sub["Mean_abs_SHAP_overall"]
    )

    plt.xlabel("Mean |SHAP|", fontsize=12, fontweight="bold")
    plt.ylabel("Feature", fontsize=12, fontweight="bold")
    plt.title(
        f"Global SHAP importance ({MODEL_NAME}, {FEATURE_SET_NAME})",
        fontsize=14,
        fontweight="bold"
    )

    ax = plt.gca()
    bold_tick_labels(ax)

    plt.tight_layout()
    plt.savefig(out_png, bbox_inches="tight")
    plt.close()


def plot_classwise_shap_bar(classwise_imp_df, out_png, top_n=15):
    n_classes = len(CLASS_ORDER)

    fig, axes = plt.subplots(
        1,
        n_classes,
        figsize=(5.2 * n_classes, max(5.5, 0.32 * top_n)),
        dpi=300
    )

    if n_classes == 1:
        axes = [axes]

    for ax, cls in zip(axes, CLASS_ORDER):
        sub = classwise_imp_df[classwise_imp_df["Class"] == cls].copy()
        sub = sub.head(top_n)
        sub = sub.sort_values("Mean_abs_SHAP_class", ascending=True)

        ax.barh(
            sub["Display_feature"],
            sub["Mean_abs_SHAP_class"]
        )

        ax.set_title(f"{cls}-type", fontsize=13, fontweight="bold")
        ax.set_xlabel("Mean |SHAP|", fontsize=11, fontweight="bold")
        ax.tick_params(axis="y", labelsize=9)

        bold_tick_labels(ax)

    fig.suptitle(
        f"Class-wise SHAP importance ({MODEL_NAME})",
        fontsize=15,
        fontweight="bold"
    )

    plt.tight_layout()
    plt.savefig(out_png, bbox_inches="tight")
    plt.close()


def plot_classwise_beeswarm(shap_arr, X_shap, model_classes, method, out_dir):
    """
    Save one beeswarm plot for each class. These plots are recommended
    for supplementary figures.
    """
    for cls in CLASS_ORDER:
        if cls not in model_classes:
            continue

        cls_idx = list(model_classes).index(cls)

        values = shap_arr[:, :, cls_idx]

        explanation = shap.Explanation(
            values=values,
            data=X_shap.values,
            feature_names=[display_feature_name(c) for c in X_shap.columns]
        )

        plt.figure(dpi=300)
        shap.plots.beeswarm(
            explanation,
            max_display=20,
            show=False
        )

        plt.title(
            f"SHAP beeswarm for {cls}-type ({method})",
            fontsize=13,
            fontweight="bold"
        )

        ax = plt.gca()
        bold_tick_labels(ax)

        out_png = os.path.join(
            out_dir,
            f"11_SHAP_beeswarm_{method}_{cls}_type.png"
        )

        plt.tight_layout()
        plt.savefig(out_png, bbox_inches="tight")
        plt.close()


def plot_representative_dependence(
    shap_arr,
    X_shap,
    representative_features,
    feature_to_best_class,
    model_classes,
    out_png
):
    resolved_features = []

    for f in representative_features:
        c = resolve_one_feature(f, X_shap.columns)

        if c is not None and c not in resolved_features:
            resolved_features.append(c)

    if len(resolved_features) == 0:
        print("No representative features were found for dependence plots.")
        return

    n = len(resolved_features)
    ncols = 3
    nrows = int(np.ceil(n / ncols))

    fig, axes = plt.subplots(
        nrows,
        ncols,
        figsize=(5.2 * ncols, 4.2 * nrows),
        dpi=300
    )

    axes = np.asarray(axes).reshape(-1)

    for ax_idx, feature in enumerate(resolved_features):
        ax = axes[ax_idx]

        feat_idx = list(X_shap.columns).index(feature)
        cls = feature_to_best_class.get(feature, CLASS_ORDER[0])
        cls_idx = list(model_classes).index(cls)

        x = X_shap[feature].values
        y = shap_arr[:, feat_idx, cls_idx]

        ax.scatter(
            x,
            y,
            s=18,
            alpha=0.55
        )

        ax.axhline(0, linestyle="--", linewidth=1)

        ax.set_xlabel(display_feature_name(feature), fontsize=11, fontweight="bold")
        ax.set_ylabel(f"SHAP value for {cls}-type", fontsize=11, fontweight="bold")
        ax.set_title(
            f"{display_feature_name(feature)} ({cls}-type)",
            fontsize=12,
            fontweight="bold"
        )

        bold_tick_labels(ax)

    for j in range(len(resolved_features), len(axes)):
        axes[j].axis("off")

    fig.suptitle(
        "Representative SHAP dependence plots for stable novel ratios",
        fontsize=15,
        fontweight="bold"
    )

    plt.tight_layout()
    plt.savefig(out_png, bbox_inches="tight")
    plt.close()


# ============================================================
# 3. Main workflow: generate SHAP interpretation for each method
# ============================================================

all_method_summary = []

for method in SHAP_METHODS:
    print("\n" + "=" * 80)
    print(f"Starting final SHAP interpretation: {method}")
    print("=" * 80)

    method_out_dir = os.path.join(OUT_DIR, method)
    os.makedirs(method_out_dir, exist_ok=True)

    # --------------------------------------------------------
    # 3.1 Read data
    # --------------------------------------------------------

    full_df = read_oof_full_dataset(method)

    raw_features = load_stable_features_for_paper()

    resolved_features, missing_features = resolve_feature_list(
        raw_features,
        full_df.columns
    )

    print("\nStable-feature matching results:")
    print("Original stable-feature count:", len(raw_features))
    print("Matched feature count:", len(resolved_features))
    print("Missing feature count:", len(missing_features))

    if missing_features:
        print("First 20 missing features:")
        print(missing_features[:20])

    if len(resolved_features) == 0:
        raise ValueError("No stable features were matched in the dataset.")

    X_raw = clean_X(full_df, resolved_features)
    X_clean, kept_features = remove_bad_features(X_raw)

    y = full_df[TYPE_COL].astype(str).values

    print("\nFinal feature count used for SHAP:", len(kept_features))
    print("Class distribution:")
    print(pd.Series(y).value_counts())

    # --------------------------------------------------------
    # 3.2 Fit the final interpretation model on the full dataset
    # --------------------------------------------------------

    imputer = SimpleImputer(strategy="median")

    X_imp_arr = imputer.fit_transform(X_clean)

    X_imp = pd.DataFrame(
        X_imp_arr,
        columns=X_clean.columns,
        index=X_clean.index
    )

    model = make_final_model()
    model.fit(X_imp, y)

    y_fit_pred = model.predict(X_imp)

    fit_acc = accuracy_score(y, y_fit_pred)
    fit_bacc = balanced_accuracy_score(y, y_fit_pred)
    fit_macro_p, fit_macro_r, fit_macro_f1, _ = precision_recall_fscore_support(
        y,
        y_fit_pred,
        average="macro",
        zero_division=0
    )

    print(
        "\nNote: the following values are apparent fit metrics from the final refit. "
        "They are used only to confirm that the model has been fitted and must not "
        "be reported as generalization performance."
    )
    print(f"Apparent Accuracy: {fit_acc:.4f}")
    print(f"Apparent Balanced accuracy: {fit_bacc:.4f}")
    print(f"Apparent Macro-F1: {fit_macro_f1:.4f}")
    print("Model class order:", model.classes_)

    # --------------------------------------------------------
    # 3.3 Select samples for SHAP calculation
    # --------------------------------------------------------

    if SHAP_MAX_SAMPLES is not None and len(X_imp) > SHAP_MAX_SAMPLES:
        rng = np.random.default_rng(RANDOM_STATE)
        sample_idx = rng.choice(
            np.arange(len(X_imp)),
            size=SHAP_MAX_SAMPLES,
            replace=False
        )
        sample_idx = np.sort(sample_idx)

        X_shap = X_imp.iloc[sample_idx].copy()
        y_shap = pd.Series(y).iloc[sample_idx].values
    else:
        X_shap = X_imp.copy()
        y_shap = y.copy()

    print("\nNumber of samples used for SHAP:", len(X_shap))

    # --------------------------------------------------------
    # 3.4 Calculate SHAP values
    # --------------------------------------------------------

    explainer = shap.TreeExplainer(model)

    raw_shap_values = explainer.shap_values(X_shap)

    shap_arr = normalize_shap_values(
        raw_shap_values,
        n_samples=X_shap.shape[0],
        n_features=X_shap.shape[1],
        n_classes=len(model.classes_)
    )

    print("SHAP array shape:", shap_arr.shape)

    # --------------------------------------------------------
    # 3.5 Calculate global and class-wise SHAP importance
    # --------------------------------------------------------

    mean_abs_overall = np.abs(shap_arr).mean(axis=(0, 2))

    global_imp_df = pd.DataFrame({
        "Feature": X_shap.columns,
        "Display_feature": [display_feature_name(c) for c in X_shap.columns],
        "Mean_abs_SHAP_overall": mean_abs_overall,
    })

    global_imp_df["Feature_group"] = global_imp_df["Feature"].apply(classify_feature_group)
    global_imp_df["Interpretation_note"] = global_imp_df["Feature"].apply(interpretation_note)

    global_imp_df = global_imp_df.sort_values(
        "Mean_abs_SHAP_overall",
        ascending=False
    ).reset_index(drop=True)

    global_imp_df["Global_rank"] = np.arange(1, len(global_imp_df) + 1)

    classwise_rows = []

    for cls in CLASS_ORDER:
        if cls not in model.classes_:
            continue

        cls_idx = list(model.classes_).index(cls)

        mean_abs_cls = np.abs(shap_arr[:, :, cls_idx]).mean(axis=0)

        for feature, imp in zip(X_shap.columns, mean_abs_cls):
            classwise_rows.append({
                "Class": cls,
                "Feature": feature,
                "Display_feature": display_feature_name(feature),
                "Mean_abs_SHAP_class": imp,
                "Feature_group": classify_feature_group(feature),
                "Interpretation_note": interpretation_note(feature),
            })

    classwise_imp_df = pd.DataFrame(classwise_rows)

    classwise_imp_df = (
        classwise_imp_df
        .sort_values(["Class", "Mean_abs_SHAP_class"], ascending=[True, False])
        .reset_index(drop=True)
    )

    classwise_imp_df["Class_rank"] = (
        classwise_imp_df
        .groupby("Class")["Mean_abs_SHAP_class"]
        .rank(method="first", ascending=False)
        .astype(int)
    )

    # Identify the class with the largest SHAP contribution for each feature.
    feature_to_best_class = {}

    for feature in X_shap.columns:
        sub = classwise_imp_df[classwise_imp_df["Feature"] == feature].copy()

        if not sub.empty:
            best_cls = (
                sub
                .sort_values("Mean_abs_SHAP_class", ascending=False)
                .iloc[0]["Class"]
            )
            feature_to_best_class[feature] = best_cls

    # --------------------------------------------------------
    # 3.6 Geochemical interpretation table
    # --------------------------------------------------------

    interpretation_df = global_imp_df.copy()

    interpretation_df["Use_in_text"] = np.where(
        interpretation_df["Global_rank"] <= TOP_N_GLOBAL,
        "Main text candidate",
        "Supplementary"
    )

    interpretation_df["Recommended_discussion_level"] = interpretation_df["Display_feature"].apply(
        lambda x: (
            "Core discussion"
            if x in [
                "10000×Ga/Al",
                "A/CNK",
                "Zr+Nb+Ce+Y",
                "Yb/Lu",
                "Dy/Ho",
                "Nb/Pb",
                "TiO2/P2O5",
                "Fe2O3t/P2O5",
                "Al2O3/K2O",
            ]
            else "Secondary discussion"
        )
    )

    # --------------------------------------------------------
    # 3.7 Save tables
    # --------------------------------------------------------

    model_info_df = pd.DataFrame({
        "Item": [
            "Model",
            "Feature_set",
            "Imputation_method",
            "N_samples_for_final_refit",
            "N_samples_for_SHAP",
            "N_features",
            "Class_order_in_model",
            "Apparent_accuracy_not_for_generalization",
            "Apparent_balanced_accuracy_not_for_generalization",
            "Apparent_macro_F1_not_for_generalization",
            "Note",
        ],
        "Value": [
            MODEL_NAME,
            FEATURE_SET_NAME,
            method,
            len(X_imp),
            len(X_shap),
            X_shap.shape[1],
            ", ".join(model.classes_),
            fit_acc,
            fit_bacc,
            fit_macro_f1,
            "Final full-data refit is used only for interpretation and visualization; cross-validated performance should be reported from the leakage-free outer-fold workflow.",
        ]
    })

    out_xlsx = os.path.join(
        method_out_dir,
        f"11_final_SHAP_importance_tables_{method}.xlsx"
    )

    with pd.ExcelWriter(out_xlsx, engine="openpyxl") as writer:
        model_info_df.to_excel(writer, sheet_name="model_info", index=False)
        global_imp_df.to_excel(writer, sheet_name="global_SHAP_importance", index=False)
        classwise_imp_df.to_excel(writer, sheet_name="classwise_SHAP_importance", index=False)
        interpretation_df.to_excel(writer, sheet_name="geochemical_interpretation", index=False)

        pd.DataFrame({
            "Missing_feature": missing_features
        }).to_excel(writer, sheet_name="missing_features", index=False)

    print("\nSHAP tables saved to:", out_xlsx)

    # --------------------------------------------------------
    # 3.8 Save figures
    # --------------------------------------------------------

    global_bar_png = os.path.join(
        method_out_dir,
        f"11_SHAP_global_bar_{method}.png"
    )

    plot_global_shap_bar(
        global_imp_df,
        global_bar_png,
        top_n=TOP_N_GLOBAL
    )

    classwise_bar_png = os.path.join(
        method_out_dir,
        f"11_SHAP_classwise_bar_{method}.png"
    )

    plot_classwise_shap_bar(
        classwise_imp_df,
        classwise_bar_png,
        top_n=TOP_N_CLASSWISE
    )

    # Beeswarm plots are recommended for supplementary figures.
    plot_classwise_beeswarm(
        shap_arr,
        X_shap,
        model.classes_,
        method,
        method_out_dir
    )

    dependence_png = os.path.join(
        method_out_dir,
        f"11_SHAP_representative_dependence_2x3_{method}.png"
    )

    plot_representative_dependence(
        shap_arr,
        X_shap,
        REPRESENTATIVE_FEATURES,
        feature_to_best_class,
        model.classes_,
        dependence_png
    )

    # --------------------------------------------------------
    # 3.9 Summary
    # --------------------------------------------------------

    top_global = global_imp_df.head(TOP_N_GLOBAL).copy()
    top_global["Imputation_method"] = method

    all_method_summary.append(top_global)

    print("\nTop 20 global SHAP features:")
    show_cols = [
        "Global_rank",
        "Display_feature",
        "Feature_group",
        "Mean_abs_SHAP_overall",
        "Interpretation_note"
    ]
    print(global_imp_df[show_cols].head(TOP_N_GLOBAL).to_string(index=False))


# ============================================================
# 4. Summary across imputation methods
# ============================================================

if all_method_summary:
    combined_top_df = pd.concat(all_method_summary, axis=0, ignore_index=True)

    combined_xlsx = os.path.join(
        OUT_DIR,
        "11_combined_top_SHAP_features.xlsx"
    )

    with pd.ExcelWriter(combined_xlsx, engine="openpyxl") as writer:
        combined_top_df.to_excel(
            writer,
            sheet_name="combined_top_features",
            index=False
        )

    print("\n========== Script 11 final SHAP interpretation completed ==========")
    print("Output directory:", OUT_DIR)
    print("Combined table:", combined_xlsx)
else:
    print("No SHAP summary results were generated.")